# Especialização do planejamento com Qwen3.5-0.8B

Neste experimento, investigo se o ajuste com LoRA melhora a proporção de planos que seguem a estrutura esperada pelo MiniCode. Comparo o Qwen3.5-0.8B antes e depois do treinamento, usando o mesmo conjunto de teste e os mesmos critérios de avaliação.

O dataset contém 80 exemplos de treino, 10 de validação e 10 de teste. Reservo a validação para acompanhar a perda durante o treinamento e o teste para a comparação entre o modelo base e o ajustado. Como os conjuntos são pequenos e sintéticos, trato os resultados como exploratórios.

Uso quantização de 4 bits e adaptadores LoRA para reduzir o consumo de memória. A execução deste notebook depende de uma GPU com CUDA. A origem dos dados, suas limitações e a rubrica de revisão estão descritas em `notebooks/data/README.md`.

Treino por 10 épocas e seleciono o adaptador pela menor loss de validação. Mantenho um único checkpoint em disco, sobrescrito quando a validação melhora. Ao final, restauro esse adaptador para a avaliação no teste. Na memória, mantenho apenas o modelo em uso e o histórico numérico das perdas, sem uma cópia dos pesos por época.

Instalo `causal-conv1d` e `flash-linear-attention[cuda]` antes de carregar o modelo para disponibilizar os kernels otimizados da convolução causal e da atenção linear. Preservo as versões de PyTorch e Triton presentes no ambiente e limito a compilação a dois processos. Uma sessão que já importou Transformers é bloqueada nessa etapa para evitar detecções de dependências desatualizadas.


In [ ]:
import os
import subprocess
import sys
import tempfile
from importlib.metadata import PackageNotFoundError, version
from pathlib import Path

# Instalo os kernels antes de importar Transformers, que mantém detecções em cache.
if 'transformers' in sys.modules:
    raise RuntimeError('Transformers já foi importado nesta sessão. A instalação dos kernels parte de uma sessão reiniciada.')

# Preservo as versões de PyTorch e Triton do ambiente para evitar conflitos de CUDA.
constraints = ['torch==' + version('torch')]
try:
    constraints.append('triton==' + version('triton'))
except PackageNotFoundError:
    pass
build_environment = os.environ.copy()
build_environment['MAX_JOBS'] = '2'
with tempfile.TemporaryDirectory(prefix='minicode-kernels-') as dependency_dir:
    constraint_path = Path(dependency_dir) / 'constraints.txt'
    constraint_path.write_text('\n'.join(constraints) + '\n', encoding='utf-8')
    pip_command = [sys.executable, '-m', 'pip', 'install', '-q', '--constraint', str(constraint_path)]
    subprocess.run(pip_command + [
        'transformers @ git+https://github.com/huggingface/transformers.git@main',
        'accelerate', 'peft', 'bitsandbytes', 'matplotlib',
        'ninja', 'packaging', 'setuptools', 'wheel',
    ], check=True, env=build_environment)
    subprocess.run(pip_command + ['flash-linear-attention[cuda]'], check=True, env=build_environment)
    subprocess.run(pip_command + ['--no-build-isolation', 'causal-conv1d'], check=True, env=build_environment)

# Verifico os imports em um processo novo, sem reutilizar módulos em cache.
subprocess.run([sys.executable, '-c', (
    'from causal_conv1d import causal_conv1d_fn, causal_conv1d_update; '
    'from fla.ops.gated_delta_rule import chunk_gated_delta_rule, fused_recurrent_gated_delta_rule; '
    'assert all(callable(fn) for fn in (causal_conv1d_fn, causal_conv1d_update, '
    'chunk_gated_delta_rule, fused_recurrent_gated_delta_rule)); '
    'print("Kernels de convolução causal e atenção linear disponíveis.")'
)], check=True, env=build_environment)


In [ ]:
import json, subprocess, sys, torch, matplotlib.pyplot as plt
from pathlib import Path
from transformers import AutoProcessor, AutoModelForMultimodalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, set_peft_model_state_dict
from peft.utils.save_and_load import load_peft_weights

REPO = Path('/content/minicode')
BRANCH = 'feat/parte-2-qwen-planejamento'
if not REPO.exists():
    subprocess.run(['git', 'clone', 'https://github.com/joaovicentedev/minicode.git', str(REPO)], check=True)
subprocess.run(['git', '-C', str(REPO), 'fetch', 'origin', BRANCH], check=True)
subprocess.run(['git', '-C', str(REPO), 'checkout', '-B', BRANCH, 'FETCH_HEAD'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(REPO)], check=True)
if not (REPO / 'notebooks/data/train.jsonl').exists():
    raise FileNotFoundError('Dataset de treinamento ausente na revisão carregada do repositório.')
sys.path.insert(0, str(REPO / 'src'))
from minicode.planning_data import load_examples, format_example, plan_score, summarize_plans
train = load_examples(REPO / 'notebooks/data/train.jsonl')
validation = load_examples(REPO / 'notebooks/data/validation.jsonl')
test = load_examples(REPO / 'notebooks/data/test.jsonl')
rubrics = json.loads((REPO / 'notebooks/data/rubrics.json').read_text(encoding='utf-8'))
print('Tamanhos:', len(train), len(validation), len(test))
assert (len(train), len(validation), len(test)) == (80, 10, 10)
assert torch.cuda.is_available(), 'GPU com CUDA indisponível neste ambiente.'


In [ ]:
MODEL_ID = 'Qwen/Qwen3.5-0.8B'
processor = AutoProcessor.from_pretrained(MODEL_ID)
tokenizer = processor.tokenizer
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token
quantization = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4', bnb_4bit_compute_dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16)
model = AutoModelForMultimodalLM.from_pretrained(MODEL_ID, quantization_config=quantization, device_map='auto')
model.config.use_cache = False


In [ ]:
def encode(example):
    pair = format_example(example)
    prefix = tokenizer.apply_chat_template([{'role': 'user', 'content': pair['prompt']}], tokenize=False, add_generation_prompt=True, enable_thinking=False)
    prompt_ids = tokenizer(prefix, add_special_tokens=False)['input_ids']
    answer_ids = tokenizer(pair['completion'] + tokenizer.eos_token, add_special_tokens=False)['input_ids']
    ids = (prompt_ids + answer_ids)[:1536]
    labels = [-100] * min(len(prompt_ids), len(ids)) + ids[len(prompt_ids):]
    return {'input_ids': ids, 'labels': labels}

encoded_train = [encode(item) for item in train]
encoded_validation = [encode(item) for item in validation]
assert all(any(label != -100 for label in row['labels']) for row in encoded_train + encoded_validation)

def batch(rows):
    width = max(len(row['input_ids']) for row in rows)
    ids = [row['input_ids'] + [tokenizer.pad_token_id] * (width - len(row['input_ids'])) for row in rows]
    labels = [row['labels'] + [-100] * (width - len(row['labels'])) for row in rows]
    return {'input_ids': torch.tensor(ids, device=model.device), 'attention_mask': torch.tensor([[int(token != tokenizer.pad_token_id) for token in line] for line in ids], device=model.device), 'labels': torch.tensor(labels, device=model.device)}


## Avaliação dos planos

Avalio o formato e o conteúdo dos planos separadamente. A métrica `structural_pass` reúne os critérios de estrutura. Para o conteúdo, uso grupos de termos definidos por tarefa em `rubrics.json`: `task_alignment` indica evidências de adequação à tarefa, `spec_coverage` indica cobertura dos critérios selecionados e `context_consistency` verifica a referência ao arquivo esperado. A métrica `quality_pass` reúne essas três dimensões, enquanto `overall_pass` combina estrutura e evidências de conteúdo.

Essas verificações são aproximações lexicais, não uma medida direta de correção semântica. Um sinônimo pode não ser reconhecido, e um plano contraditório pode conter todos os termos esperados. Por isso, salvo as respostas em `predictions.json`, junto com as notas automáticas e campos para a revisão manual descrita em `notebooks/data/README.md`.

As rubricas ficam restritas à avaliação e não entram nos prompts nem no treinamento. Uso os mesmos critérios para o modelo base e o ajustado, sem alterá-los em função das respostas de teste.


In [ ]:
@torch.no_grad()
def generate_plans(active_model, examples):
    active_model.eval()
    answers = []
    for example in examples:
        prompt = format_example(example)['prompt']
        text = tokenizer.apply_chat_template([{'role': 'user', 'content': prompt}], tokenize=False, add_generation_prompt=True, enable_thinking=False)
        inputs = tokenizer(text, return_tensors='pt').to(active_model.device)
        output = active_model.generate(**inputs, max_new_tokens=160, do_sample=False, pad_token_id=tokenizer.eos_token_id)
        answers.append(tokenizer.decode(output[0, inputs['input_ids'].shape[1]:], skip_special_tokens=True))
    return answers

# Avalio o modelo base no conjunto de teste antes do ajuste dos pesos.
base_answers = generate_plans(model, test)
base_metrics = summarize_plans(base_answers, test, rubrics)
print('Base:', base_metrics)


In [ ]:
import math

# Uma nova execução do treino parte do modelo base, sem adaptador prévio.
if getattr(model, 'peft_config', None):
    raise RuntimeError('O modelo já contém LoRA. Uma nova execução completa parte de uma sessão reiniciada.')

EPOCHS = 10
OUTPUT = Path('/content/minicode-plan-lora')
OUTPUT.mkdir(parents=True, exist_ok=True)
lora = LoraConfig(r=8, lora_alpha=16, lora_dropout=0.05, bias='none', target_modules=['q_proj', 'v_proj'], task_type='CAUSAL_LM')
model = get_peft_model(model, lora)
model.print_trainable_parameters()
optimizer = torch.optim.AdamW((p for p in model.parameters() if p.requires_grad), lr=2e-4)
train_loss, validation_loss = [], []
best_validation_loss = float('inf')
best_epoch = None
for epoch in range(EPOCHS):
    model.train()
    total_train_loss = 0.0
    for row in encoded_train:
        optimizer.zero_grad(set_to_none=True)
        loss = model(**batch([row])).loss
        loss.backward()
        optimizer.step()
        total_train_loss += loss.item()
        del loss
    train_loss.append(total_train_loss / len(encoded_train))
    optimizer.zero_grad(set_to_none=True)
    model.eval()
    total_validation_loss = 0.0
    with torch.no_grad():
        for row in encoded_validation:
            loss = model(**batch([row])).loss
            total_validation_loss += loss.item()
            del loss
    validation_loss.append(total_validation_loss / len(encoded_validation))
    if not math.isfinite(train_loss[-1]) or not math.isfinite(validation_loss[-1]):
        raise RuntimeError('Loss não finita durante o treinamento; o último checkpoint válido permanece no disco.')
    if validation_loss[-1] < best_validation_loss:
        best_validation_loss = validation_loss[-1]
        best_epoch = epoch + 1
        # Sobrescrevo o mesmo adaptador em disco, sem cópias dos pesos por época.
        model.save_pretrained(OUTPUT, safe_serialization=True)
        processor.save_pretrained(OUTPUT)
    print(epoch + 1, train_loss[-1], validation_loss[-1])
    (OUTPUT / 'training_history.json').write_text(json.dumps({
        'epochs_completed': epoch + 1, 'best_epoch': best_epoch,
        'best_validation_loss': best_validation_loss,
        'train_loss': train_loss, 'validation_loss': validation_loss,
    }, indent=2), encoding='utf-8')

# Libero o estado do otimizador antes da inferência e restauro apenas o melhor LoRA.
del optimizer
if best_epoch is None:
    raise RuntimeError('Nenhuma época produziu loss de validação finita.')
best_adapter = load_peft_weights(str(OUTPUT), device='cpu')
set_peft_model_state_dict(model, best_adapter)
del best_adapter
model.eval()
torch.cuda.empty_cache()
print('Melhor época:', best_epoch, '| Loss de validação:', best_validation_loss)


In [ ]:
plt.plot(range(1, len(train_loss)+1), train_loss, label='Treino')
plt.plot(range(1, len(validation_loss)+1), validation_loss, label='Validação')
plt.xlabel('Época'); plt.ylabel('Loss'); plt.legend(); plt.show()
ft_answers = generate_plans(model, test)
ft_metrics = summarize_plans(ft_answers, test, rubrics)
print('Base:', base_metrics)
print('LoRA:', ft_metrics)
for example, before, after in zip(test, base_answers, ft_answers):
    print('\nTarefa:', example['task'], '\nBase:', before, '\nLoRA:', after)


In [ ]:
(OUTPUT / 'metrics.json').write_text(json.dumps({'base': base_metrics, 'lora': ft_metrics, 'train_loss': train_loss, 'validation_loss': validation_loss, 'best_epoch': best_epoch, 'best_validation_loss': best_validation_loss, 'epochs': EPOCHS}, indent=2), encoding='utf-8')
print('Adaptador e métricas:', OUTPUT)

# Salvo as respostas e as notas por tarefa como apoio à revisão manual.
review = []
for example, before, after in zip(test, base_answers, ft_answers, strict=True):
    review.append({
        'example': example, 'rubric': rubrics[example['task']],
        'base': {'plan': before, 'scores': plan_score(before, rubrics[example['task']])},
        'lora': {'plan': after, 'scores': plan_score(after, rubrics[example['task']])},
        'manual_review': {'base': None, 'lora': None},
    })
(OUTPUT / 'predictions.json').write_text(json.dumps(review, ensure_ascii=False, indent=2), encoding='utf-8')


## Integração com o MiniCode

A avaliação no agente completo fica como uma etapa posterior ao treinamento. A comparação considera o modelo base e o ajustado em endpoints compatíveis com OpenAI, selecionados por `MINICODE_PLAN_MODEL`. Em cada configuração, `minicode-eval run` executa os mesmos casos e `minicode-eval grade` avalia as soluções com testes externos.

Além das métricas locais dos planos, a análise considera a `TaskSuccessRate` do agente. A referência adicional é o fluxo padrão do MiniCode, sem endpoint separado para planejamento. A comparação entre esse fluxo e o planejamento separado permite investigar o efeito dessa configuração no sistema completo.

Os relatórios dessas execuções ainda não estão disponíveis neste notebook. Os resultados locais do treinamento, por si só, não demonstram ganho no agente completo.
